# eDNA Metabarcoding Pipeline Analysis: Lake Geneva (eKOI Database)
### A Reproducible Workflow for MinION Amplicon Sequencing (18S & COI)
**Project:** Genorobotics Semester Project (EPFL)

**Markers:** 18S rRNA (SILVA) & COI (eKOI PR2)

---

## 0. Critical Review: Database Changes

### COI: eKOI PR2 replaces MIDORI2
The COI marker now uses the **eKOI PR2 database** (15,947 sequences) instead of MIDORI2 (2.2M+ sequences).
* **Broader eukaryotic coverage:** eKOI includes protists, fungi, diatoms, and plants — not just metazoans.
* **PR2 taxonomy:** Clean 9-level hierarchy without NCBI taxon ID artifacts.
* **Smaller database:** 66MB vs 9.4GB — dramatically faster SINTAX queries.
* **Trade-off:** Fewer reference sequences means potentially lower species-level resolution for metazoans.

### 18S: SILVA unchanged
The 18S marker continues to use the SILVA v123 database.

### eKOI Taxonomy Mapping
| SINTAX level | PR2 field | Example |
|---|---|---|
| `d:` Domain | Kingdom | Eukaryota |
| `k:` Kingdom | Supergroup | Obazoa, TSAR, Archaeplastida |
| `p:` Phylum | Division | Opisthokonta, Alveolata |
| `c:` Class | Subdivision | Metazoa, Fungi, Apicomplexa |
| `o:` Order | Class | Insecta, Teleostei |
| `f:` Family | Family | resolved from Order/Family |
| `g:` Genus | Genus | Daphnia |
| `s:` Species | Species | Daphnia_galeata |

### Key Question: Does eKOI improve COI ciliate classification?
The previous MIDORI2 analysis showed that ciliates dominated the COI signal but were poorly classified. eKOI's protist coverage should help identify these organisms more accurately.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from pathlib import Path
from Bio import SeqIO
from matplotlib import cm

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

def clean_sample_names(columns):
    return [c.replace('Sample_barcode', '').replace('Sample_', '') for c in columns]

def get_tax_prefix(df):
    for col in df.columns:
        if col.startswith("SILVA_"):
            return "SILVA"
        if col.startswith("MIDORI_"):
            return "MIDORI"
        if col.startswith("eKOI_"):
            return "eKOI"
    return "SILVA"

BASE = Path("out/Water_eDNA_18S_COI_14_01_26")

## Table Headers & Data Structure
Inspect the comprehensive taxonomy CSV files to verify column names and data shape.

In [ ]:
df_18s = pd.read_csv(BASE / 'taxonomy_summary/comprehensive_taxonomy_18S.csv')
df_coi_raw = pd.read_csv(BASE / 'taxonomy_summary/comprehensive_taxonomy_COI.csv')

prefix_18s = get_tax_prefix(df_18s)
prefix_coi = get_tax_prefix(df_coi_raw)

for label, df, pfx in [("18S", df_18s, prefix_18s), ("COI", df_coi_raw, prefix_coi)]:
    print(f"\n{'='*60}")
    print(f"  {label} ({pfx} database)  \u2014  {df.shape[0]} OTUs \u00d7 {df.shape[1]} columns")
    print(f"{'='*60}")
    
    sample_cols = [c for c in df.columns if c.startswith("Sample_")]
    taxonomy_cols = [c for c in df.columns if c.startswith(f"{pfx}_") or c.startswith("NCBI_")]
    meta_cols = [c for c in df.columns if c not in sample_cols + taxonomy_cols]
    
    print(f"\nMetadata columns:  {meta_cols}")
    print(f"Sample columns:    {sample_cols}")
    print(f"Taxonomy columns:  {taxonomy_cols}")
    print(f"\nFirst 3 rows:")
    display(df.head(3))

## Confidence Distribution DashboardVisualize the SINTAX bootstrap confidence for taxonomy assignments across ranks.Only assignments with confidence >= 0.8 are stored in the CSV.

In [ ]:
# Confidence Distribution Dashboard
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

marker_info = [
    ("18S", df_18s, prefix_18s, "#2196F3"),
    ("COI", df_coi_raw, prefix_coi, "#4CAF50"),
]

# Check if confidence columns exist
has_conf = any(c.endswith('_Conf') for c in df_18s.columns) or any(c.endswith('_Conf') for c in df_coi_raw.columns)

if not has_conf:
    print("No confidence columns found in CSV files.")
    print("Regenerate with: bash regenerate_taxonomy.sh")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for row, (marker, df, pfx, color) in enumerate(marker_info):
        # Left: Mean confidence per rank
        ax = axes[row, 0]
        mean_confs = []
        for rank in ranks:
            conf_col = f'{pfx}_{rank}_Conf'
            if conf_col in df.columns:
                vals = pd.to_numeric(df[conf_col], errors='coerce').dropna()
                mean_confs.append(vals.mean() if len(vals) > 0 else 0)
            else:
                mean_confs.append(0)
        
        bar_colors = [color if c >= 0.8 else '#e74c3c' for c in mean_confs]
        bars = ax.bar(ranks, mean_confs, color=bar_colors, alpha=0.85, edgecolor='white')
        ax.axhline(y=0.8, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Threshold (0.8)')
        ax.set_title(f'{marker} ({pfx}) - Mean Confidence per Rank', fontweight='bold')
        ax.set_ylabel('Mean Bootstrap Confidence')
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis='x', rotation=45)
        for bar, val in zip(bars, mean_confs):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                        f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.legend(fontsize=9)
        
        # Right: Confidence histogram for best rank
        ax2 = axes[row, 1]
        hist_rank = 'Genus' if pfx == 'SILVA' else 'Phylum'
        conf_col = f'{pfx}_{hist_rank}_Conf'
        if conf_col in df.columns:
            vals = pd.to_numeric(df[conf_col], errors='coerce').dropna()
            if len(vals) > 0:
                ax2.hist(vals, bins=20, color=color, alpha=0.75, edgecolor='white')
                ax2.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Threshold (0.8)')
                n_above = (vals >= 0.8).sum()
                ax2.set_title(f'{marker} - {hist_rank} Confidence Distribution\n'
                             f'({n_above}/{len(vals)} assignments >= 0.8)', fontweight='bold')
            else:
                ax2.set_title(f'{marker} - No {hist_rank} assignments', fontweight='bold')
        else:
            ax2.text(0.5, 0.5, f'No {conf_col} column', transform=ax2.transAxes, ha='center')
            ax2.set_title(f'{marker} - {hist_rank} Confidence', fontweight='bold')
        ax2.set_xlabel('Bootstrap Confidence')
        ax2.set_ylabel('Number of OTUs')
        ax2.legend(fontsize=9)
    
    plt.suptitle('SINTAX Confidence Overview (Water Dataset)', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


## Raw Read Length Distributions (Pre-Clustering)

In [ ]:
import gzip

barcode_dirs = sorted(BASE.glob("barcode*"))
marker_lengths = {"18S": [], "COI": []}
marker_colors_raw = {"18S": "#2196F3", "COI": "#4CAF50"}

for bd in barcode_dirs:
    for marker in marker_lengths:
        fq = bd / f"filtered_reads_{marker}.fastq.gz"
        if fq.exists():
            with gzip.open(str(fq), 'rt') as f:
                for i, line in enumerate(f):
                    if i % 4 == 1:
                        marker_lengths[marker].append(len(line.strip()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, lengths) in enumerate(marker_lengths.items()):
    ax = axes[i]
    if lengths:
        ax.hist(lengths, bins=100, color=marker_colors_raw[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths):,} reads", fontsize=13, fontweight="bold")
        ax.set_xlabel("Read length (bp)")
        ax.set_ylabel("Number of reads")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths):,} reads, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nNo reads found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Raw Read Length Distributions (Water)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Consensus OTU Sequence Length Distributions

In [ ]:
fasta_files = {
    "18S": BASE / "temp_clustering/consensus_18S_clean.fasta",
    "COI": BASE / "temp_clustering/consensus_COI_clean.fasta",
}
marker_colors = {"18S": "#2196F3", "COI": "#4CAF50"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, path) in enumerate(fasta_files.items()):
    ax = axes[i]
    if path.exists():
        lengths = [len(rec.seq) for rec in SeqIO.parse(str(path), "fasta")]
        ax.hist(lengths, bins=50, color=marker_colors[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths)} OTUs", fontsize=13, fontweight="bold")
        ax.set_xlabel("Sequence length (bp)")
        ax.set_ylabel("Number of OTUs")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths)} OTUs, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nFASTA not found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Consensus OTU Sequence Length Distributions (Water)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
# Part A: 18S Marker Biodiversity Analysis (SILVA)
*Objective: Characterize the planktonic community using the 18S marker. This section is unchanged from the MIDORI2 analysis since 18S still uses SILVA.*

## A.1a Broad Taxonomic Structure (18S)

In [ ]:
sample_cols_18s = [c for c in df_18s.columns if c.startswith('Sample_') and 'unclassified' not in c]
phylum_col_18s = f'{prefix_18s}_Phylum'

df_18s[phylum_col_18s] = df_18s[phylum_col_18s].fillna('Unassigned')
phylum_18s = df_18s.groupby(phylum_col_18s)[sample_cols_18s].sum()

phylum_18s['Total'] = phylum_18s.sum(axis=1)
phylum_18s = phylum_18s.sort_values('Total', ascending=False)
top_phyla = phylum_18s.head(10).index

plot_data = phylum_18s.loc[top_phyla].drop(columns='Total')
others = phylum_18s.loc[~phylum_18s.index.isin(top_phyla)].drop(columns='Total').sum()
plot_data.loc['Others'] = others

fig, ax = plt.subplots(figsize=(12, 6))
plot_data.columns = clean_sample_names(plot_data.columns)
plot_data.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Lake Community Composition \u2014 Phylum (18S, {prefix_18s})', fontweight='bold')
ax.set_xlabel('Sample ID')
ax.set_ylabel('Relative Abundance')
ax.legend(title='Phylum', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## A.1b Class-Level Breakdown (18S)

In [ ]:
class_col_18s = f'{prefix_18s}_Class'
df_18s[class_col_18s] = df_18s[class_col_18s].fillna('Unassigned')
class_18s = df_18s.groupby(class_col_18s)[sample_cols_18s].sum()

class_18s['Total'] = class_18s.sum(axis=1)
class_18s = class_18s.sort_values('Total', ascending=False)
top_classes = class_18s.head(15).index

plot_cls = class_18s.loc[top_classes].drop(columns='Total')
others_cls = class_18s.loc[~class_18s.index.isin(top_classes)].drop(columns='Total').sum()
plot_cls.loc['Others'] = others_cls

num_colors = len(plot_cls)
colors = cm.tab20(np.linspace(0, 1, num_colors))
custom_colors = []
for i, cls_name in enumerate(plot_cls.index):
    if cls_name == 'Unassigned':
        custom_colors.append('#D3D3D3')
    elif cls_name == 'Others':
        custom_colors.append('#696969')
    else:
        custom_colors.append(colors[i])

fig, ax = plt.subplots(figsize=(12, 7))
plot_cls.columns = clean_sample_names(plot_cls.columns)
plot_cls.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=custom_colors)

ax.set_title(f'Class-Level Composition (18S, {prefix_18s})', fontweight='bold')
ax.set_ylabel('Relative Abundance')
handles, labels = ax.get_legend_handles_labels()
ax.legend(reversed(handles), reversed(labels), bbox_to_anchor=(1.02, 1), loc='upper left', title='Class')

plt.tight_layout()
plt.show()

## A.2 Taxonomic Resolution Decay (18S)

In [ ]:
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']
levels_18s = [f'{prefix_18s}_{r}' for r in ranks]
unassigned_counts_18s = []

for level in levels_18s:
    n_unassigned = df_18s[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_18s)) * 100
    unassigned_counts_18s.append(pct)

plt.figure(figsize=(8, 5))
sns.barplot(x=ranks, y=unassigned_counts_18s, color="salmon")
plt.title(f'Taxonomic Resolution Decay (18S, {prefix_18s})', fontweight='bold')
plt.ylabel('% of OTUs Unassigned')
plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

---
# Part B: COI Marker Biodiversity Analysis (eKOI)
*Objective: Characterize the metazoan/eukaryotic community using COI with the eKOI PR2 database.*

### Key changes from MIDORI2:
* eKOI taxonomy is cleaner — no NCBI taxon ID artifacts, no rank prefix stripping needed.
* "Phylum" = PR2 Division (Opisthokonta, Stramenopiles, Alveolata)
* "Class" = PR2 Subdivision (Metazoa, Fungi, Apicomplexa)
* Better protist/ciliate coverage should reveal what MIDORI2 missed.

## B.1a Broad Taxonomic Structure (COI, eKOI)

In [ ]:
df_coi = df_coi_raw.copy()
sample_cols_coi = [c for c in df_coi.columns if c.startswith('Sample_') and 'unclassified' not in c]
phylum_col_coi = f'{prefix_coi}_Phylum'

df_coi[phylum_col_coi] = df_coi[phylum_col_coi].fillna('Unassigned')
phylum_coi = df_coi.groupby(phylum_col_coi)[sample_cols_coi].sum()

phylum_coi['Total'] = phylum_coi.sum(axis=1)
phylum_coi = phylum_coi.sort_values('Total', ascending=False)
top_phyla_coi = phylum_coi.head(10).index

plot_data_coi = phylum_coi.loc[top_phyla_coi].drop(columns='Total')
others_coi = phylum_coi.loc[~phylum_coi.index.isin(top_phyla_coi)].drop(columns='Total').sum()
plot_data_coi.loc['Others'] = others_coi

fig, ax = plt.subplots(figsize=(12, 6))
plot_data_coi.columns = clean_sample_names(plot_data_coi.columns)
plot_data_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Lake Community Composition \u2014 Phylum (COI, {prefix_coi})', fontweight='bold')
ax.set_xlabel('Sample ID')
ax.set_ylabel('Relative Abundance')
ax.legend(title='Phylum', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## B.1b Class-Level Breakdown (COI, eKOI)
In eKOI, "Class" = PR2 Subdivision. Look for **Metazoa**, **Fungi**, **Apicomplexa**, etc.

In [ ]:
class_col_coi = f'{prefix_coi}_Class'
df_coi[class_col_coi] = df_coi[class_col_coi].fillna('Unassigned')
class_coi = df_coi.groupby(class_col_coi)[sample_cols_coi].sum()

class_coi['Total'] = class_coi.sum(axis=1)
class_coi = class_coi.sort_values('Total', ascending=False)
top_classes_coi = class_coi.head(15).index

plot_cls_coi = class_coi.loc[top_classes_coi].drop(columns='Total')
others_cls_coi = class_coi.loc[~class_coi.index.isin(top_classes_coi)].drop(columns='Total').sum()
plot_cls_coi.loc['Others'] = others_cls_coi

num_colors_coi = len(plot_cls_coi)
colors_coi = cm.tab20(np.linspace(0, 1, num_colors_coi))
custom_colors_coi = []
for i, cls_name in enumerate(plot_cls_coi.index):
    if cls_name == 'Unassigned':
        custom_colors_coi.append('#D3D3D3')
    elif cls_name == 'Others':
        custom_colors_coi.append('#696969')
    else:
        custom_colors_coi.append(colors_coi[i])

fig, ax = plt.subplots(figsize=(12, 7))
plot_cls_coi.columns = clean_sample_names(plot_cls_coi.columns)
plot_cls_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=custom_colors_coi)

ax.set_title(f'Class-Level Composition (COI, {prefix_coi})', fontweight='bold')
ax.set_ylabel('Relative Abundance')
handles, labels = ax.get_legend_handles_labels()
ax.legend(reversed(handles), reversed(labels), bbox_to_anchor=(1.02, 1), loc='upper left', title='Class')

plt.tight_layout()
plt.show()

## B.2 Order-Level Breakdown (COI, eKOI)
In eKOI, "Order" = PR2 Class level (e.g., Insecta, Teleostei, Mammalia for Metazoa).

In [ ]:
order_col_coi = f'{prefix_coi}_Order'
df_coi[order_col_coi] = df_coi[order_col_coi].fillna('Unassigned')
order_coi = df_coi.groupby(order_col_coi)[sample_cols_coi].sum()

order_coi['Total'] = order_coi.sum(axis=1)
order_coi = order_coi.sort_values('Total', ascending=False)
top_orders_coi = order_coi.head(15).index

plot_ord_coi = order_coi.loc[top_orders_coi].drop(columns='Total')
others_ord_coi = order_coi.loc[~order_coi.index.isin(top_orders_coi)].drop(columns='Total').sum()
plot_ord_coi.loc['Others'] = others_ord_coi

fig, ax = plt.subplots(figsize=(12, 7))
plot_ord_coi.columns = clean_sample_names(plot_ord_coi.columns)
plot_ord_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Order-Level Composition (COI, {prefix_coi})', fontweight='bold')
ax.set_ylabel('Relative Abundance')
ax.legend(title='Order', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## B.3 Genus-Level Top 20 (COI, eKOI)

In [ ]:
genus_col_coi = f'{prefix_coi}_Genus'
df_coi[genus_col_coi] = df_coi[genus_col_coi].fillna('Unassigned')
genus_coi = df_coi.groupby(genus_col_coi)[sample_cols_coi].sum()
genus_coi['Total'] = genus_coi.sum(axis=1)
genus_coi = genus_coi.sort_values('Total', ascending=False)

top_genera_coi = genus_coi.head(20)

# Compute mean confidence per genus
conf_col = f'{prefix_coi}_Genus_Conf'
genus_conf = {}
if conf_col in df_coi.columns:
    for genus in top_genera_coi.index:
        mask = df_coi[genus_col_coi] == genus
        vals = pd.to_numeric(df_coi.loc[mask, conf_col], errors='coerce').dropna()
        genus_conf[genus] = vals.mean() if len(vals) > 0 else None

fig, ax = plt.subplots(figsize=(10, 8))
bars = sns.barplot(x=top_genera_coi['Total'], y=top_genera_coi.index, palette='viridis', ax=ax)
ax.set_title(f'Top 20 Genera (COI, {prefix_coi})', fontweight='bold')
ax.set_xlabel('Total Abundance')
ax.set_ylabel('Genus')

# Annotate with confidence
if genus_conf:
    for j, genus in enumerate(top_genera_coi.index):
        conf = genus_conf.get(genus)
        if conf is not None:
            ax.text(top_genera_coi.loc[genus, 'Total'] + 0.001, j,
                    f' ({conf:.2f})', va='center', fontsize=9, color='darkred', fontweight='bold')

plt.tight_layout()
plt.show()


## B.4 Taxonomic Resolution Comparison: 18S (SILVA) vs COI (eKOI)

In [ ]:
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

levels_18s = [f'{prefix_18s}_{r}' for r in ranks]
unassigned_18s = []
for level in levels_18s:
    n_unassigned = df_18s[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_18s)) * 100
    unassigned_18s.append(pct)

levels_coi = [f'{prefix_coi}_{r}' for r in ranks]
unassigned_coi = []
for level in levels_coi:
    n_unassigned = df_coi[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_coi)) * 100
    unassigned_coi.append(pct)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(ranks))
width = 0.35

bars1 = ax.bar(x - width/2, unassigned_18s, width, label=f'18S ({prefix_18s})', color='salmon', alpha=0.8)
bars2 = ax.bar(x + width/2, unassigned_coi, width, label=f'COI ({prefix_coi})', color='steelblue', alpha=0.8)

ax.set_title('Taxonomic Resolution Decay: 18S (SILVA) vs COI (eKOI)', fontweight='bold')
ax.set_ylabel('% of OTUs Unassigned')
ax.set_xticks(x)
ax.set_xticklabels(ranks, rotation=45)
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.show()

---
## Part C: eKOI Non-Metazoan Detections (COI)
Key advantage of eKOI: better classification of protists, fungi, and other non-metazoan eukaryotes that MIDORI2 missed or misclassified.

In [ ]:
class_col = f'{prefix_coi}_Class'
phylum_col = f'{prefix_coi}_Phylum'
genus_col = f'{prefix_coi}_Genus'

# Non-metazoan = Class is not Metazoa and not empty/Unassigned
non_metazoan = df_coi[
    (~df_coi[class_col].isin(['Metazoa', 'Unassigned', '', np.nan])) & 
    (df_coi[class_col].notna())
]

if len(non_metazoan) > 0:
    print(f"Non-Metazoan COI OTUs: {len(non_metazoan)} ({100*len(non_metazoan)/len(df_coi):.1f}% of total)")
    print(f"\nNon-Metazoan Classes (Subdivision):")
    
    non_meta_class = non_metazoan.groupby(class_col)[sample_cols_coi].sum()
    non_meta_class['Total'] = non_meta_class.sum(axis=1)
    non_meta_class = non_meta_class.sort_values('Total', ascending=False)
    for cls, row in non_meta_class.iterrows():
        print(f"  {cls}: {row['Total']:.4f}")
    
    print(f"\nTop 10 non-Metazoan Genera:")
    non_meta_genus = non_metazoan.groupby(genus_col)[sample_cols_coi].sum()
    non_meta_genus['Total'] = non_meta_genus.sum(axis=1)
    non_meta_genus = non_meta_genus.sort_values('Total', ascending=False)
    for gen, row in non_meta_genus.head(10).iterrows():
        print(f"  {gen}: {row['Total']:.4f}")
    
    # Plot non-metazoan classes
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=non_meta_class['Total'], y=non_meta_class.index, palette='Set2', ax=ax)
    ax.set_title(f'Non-Metazoan COI Detections (eKOI)', fontweight='bold')
    ax.set_xlabel('Total Abundance')
    ax.set_ylabel('Class (Subdivision)')
    plt.tight_layout()
    plt.show()
else:
    print("No non-Metazoan OTUs detected in COI")

---
## Part D: Cross-Marker Comparison (18S SILVA vs COI eKOI)
Compare the taxonomic profiles from both markers to identify concordance and unique detections.

In [ ]:
# Compare genus-level detections between 18S and COI
genus_col_18s = f'{prefix_18s}_Genus'
genus_col_coi = f'{prefix_coi}_Genus'

genera_18s = set(df_18s[genus_col_18s].dropna().unique()) - {'Unassigned', ''}
genera_coi = set(df_coi[genus_col_coi].dropna().unique()) - {'Unassigned', ''}

shared = genera_18s & genera_coi
only_18s = genera_18s - genera_coi
only_coi = genera_coi - genera_18s

print(f"Genera detected by 18S (SILVA): {len(genera_18s)}")
print(f"Genera detected by COI (eKOI):  {len(genera_coi)}")
print(f"Shared genera:                  {len(shared)}")
print(f"Only in 18S:                    {len(only_18s)}")
print(f"Only in COI:                    {len(only_coi)}")

if shared:
    print(f"\nShared genera (first 20): {sorted(shared)[:20]}")